Import libraires

In [ ]:
from google.colab import files
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os

Clonage du repo git sur le colab

In [ ]:
!git clone https://github.com/alextonon/Hackaton_abeilles_tigres

Téléchargement des données depuis kaggle

In [ ]:
# 1. Upload du fichier kaggle.json (choisir le bon fichier sur l'ordinateur)
files.upload()

# 2. Création du dossier caché et déplacement du fichier
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# 3. Sécurisation du fichier (obligatoire pour l'API Kaggle)
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#Téléchargement du fichier depuis kaggle
!kaggle competitions download -c reconnaissance-dabeilles-francaises

In [ ]:
#Dézipage du fichier téléchargé depuis kaggle
!unzip reconnaissance-dabeilles-francaises.zip -d Hackaton_abeilles_tigres

In [ ]:
#On se place dans le repo
%cd /content/Hackaton_abeilles_tigres/

In [ ]:
#Déplacement des classe
!rm -rf data/train/Andrena\ dilleri data/train/Andrena\ pinguis data/train/Andrena\ prodigiosa
!mv processed_classes/* Hackaton_abeilles_tigres/data/train

Création du csv train_corrected

In [ ]:
!python3 lib/data/train_csv.py

# Entraînement du modèle

# __Master Pipeline__

# **0. Librairies**

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os
from sklearn.metrics import confusion_matrix


sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset
from lib.data.preprocessing import TorchPreprocessor
from lib.data.train_val_split import train_val_split
from lib.data.preprocessing import TorchPreprocessor
from lib.data.data_augmentation import data_augmented_loader
from lib.utils.model_saver import ModelSaver

In [ ]:
## CONFIGURATION
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
NUM_EPOCHS = 10
LR = 1e-4

DROPOUT = 0.4
WEIGHT_DECAY = 1e-4

USER_NAME = 'Maignan'

NUM_CLASSES = 50

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

### **Booléen par block**

In [ ]:
scheduler = True
focal_loss = False
wheigted = False
unfreeze = False
data_augmentation = False
data_differntiation = False

## **1. Preprocessing**

In [ ]:
resnet_processor_parameters = {
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
    "crop_size": (224, 224)
}

train_preprocessor_light = TorchPreprocessor (mean = resnet_processor_parameters["mean"], 
                                              std = resnet_processor_parameters["std"], 
                                              normalize = True,
                                              augmentation = 'light', 
                                              resize_method = 'crop', 
                                              resize_value = 256, 
                                              target_size = (224,224)) 

train_preprocessor_heavy = TorchPreprocessor (mean = resnet_processor_parameters["mean"], 
                                              std = resnet_processor_parameters["std"], 
                                              normalize = True,
                                              augmentation = 'heavy', 
                                              resize_method = 'crop', 
                                              resize_value = 256, 
                                              target_size = (224,224)) 


val_preprocessor = TorchPreprocessor (mean = resnet_processor_parameters["mean"], 
                                      std = resnet_processor_parameters["std"], 
                                      normalize = True,
                                      augmentation = 'None', 
                                      resize_method = 'crop', 
                                      resize_value = 256, 
                                      target_size = (224,224)) 

In [ ]:

if data_augmentation:
    train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), batch_size=BATCH_SIZE,  apply_augmentation = data_augmentation)
elif data_augmentation and data_differntiation:
    train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), 
                                                     batch_size=BATCH_SIZE,
                                                     apply_augmentation = data_augmentation, 
                                                     distinguish_classes=data_differntiation,
                                                     train_preprocessor_light = train_preprocessor_light,
                                                     train_preprocessor_heavy = train_preprocessor_heavy,
                                                     val_preprocessor = val_preprocessor)
else:
    train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), batch_size=BATCH_SIZE)

Train prêt : 6417 images (Pas d'augmentation)
Val prête  : 1582 images (sans augmentation)


In [ ]:
# --- Récupérer tous les labels du val_loader ---
all_labels = []

for _, labels in val_loader:
    all_labels.extend(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

# --- Compter le nombre d'images par classe ---
class_counts_val = pd.Series(all_labels).value_counts().sort_index()

# --- Créer un DataFrame ---
df_val_counts = pd.DataFrame({
    "class": class_counts_val.index,
    "num_images": class_counts_val.values
})

# --- Sauvegarder dans un CSV ---
csv_val_path = "val_class_counts.csv"
df_val_counts.to_csv(csv_val_path, index=False)
print(f"CSV des images par classe dans la validation sauvegardé dans : {csv_val_path}")

CSV des images par classe dans la validation sauvegardé dans : val_class_counts.csv


## **2. Modèle**

In [ ]:
val_class_counts = pd.read_csv("val_class_counts.csv")
weights = 1.0 / val_class_counts["num_images"].values

In [ ]:
class ResnetFineTune (nn.Module):
    def __init__(self, num_classes, 
                 dropout, 
                 freeze = False):
        
        super(ResnetFineTune, self).__init__()

        self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
        if freeze:
            for param in self.resnet.parameters():
                param.requires_grad = False

            for param in self.resnet.layer4.parameters():
                param.requires_grad = True
                
        if unfreeze:
            for param in self.resnet.parameters():
                param.requires_grad = False

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )

    def forward (self, x):
        return self.resnet(x)
    
    def inference(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            probs = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
        
        return pred_class

In [ ]:
model = ResnetFineTune(NUM_CLASSES, DROPOUT)
model.to(DEVICE)

ResnetFineTune(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
    

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

        if alpha is not None:
            if not isinstance(alpha, torch.Tensor):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            self.register_buffer('alpha', alpha)
        else:
            self.alpha = None

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss).clamp(min=1e-8, max=1.0)
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.alpha is not None:
            alpha = self.alpha.to(targets.device) 
            alpha = alpha[targets]
            focal_loss = alpha * focal_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

train_labels = np.load('train_labels.npy')


def compute_alpha(train_labels, num_classes):
    classes = np.arange(num_classes)
    weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=train_labels
    )
    # Normaliser entre 0 et 1
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)



In [ ]:

if focal_loss:

    alpha = compute_alpha(train_labels, num_classes=50)
    criterion = FocalLoss(gamma=2.0, alpha=None)

elif wheigted:
    val_class_counts = pd.read_csv("val_class_counts.csv")
    weights = 1.0 / np.sqrt(val_class_counts["num_images"].values)


    criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE))
else:
    criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
ModelSaver = ModelSaver(model=model, username=USER_NAME)
ModelSaver.save_training_config(model, optimizer, BATCH_SIZE, NUM_EPOCHS, LR, DROPOUT, DEVICE, criterion=criterion)

In [ ]:
if scheduler:
    # Usage de OneCycleLR
    steps_per_epoch = len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LR,
        steps_per_epoch=steps_per_epoch,
        epochs=NUM_EPOCHS,
        pct_start=0.1, # 10% du temps d'entraînement sera dédié au Warm-up (montée douce du LR)
        div_factor=10.0, # Le LR commence à LR / 10
        final_div_factor=100.0 # Le LR finit très bas
    )

In [ ]:
def unfreeze_progressive(model, epoch):
    if epoch == 5:
        # Dégeler les derniers blocs features
        for param in model.model.features[6:].parameters():
            param.requires_grad = True
    if epoch == 10:
        # Tout dégeler
        for param in model.model.features.parameters():
            param.requires_grad = True

## **3. F1-score**

In [ ]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

In [ ]:
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

def compute_all_metrics(all_labels, all_preds, num_classes):
    """
    Calcule toutes les métriques en utilisant ta fonction compute_f1 personnalisée.
    """
    all_preds_np = np.array(all_preds)
    all_labels_np = np.array(all_labels)
    acc = np.mean(all_preds_np == all_labels_np)

    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    precision_per_class = precision_score(all_labels, all_preds, average=None, labels=range(num_classes), zero_division=0)
    recall_per_class = recall_score(all_labels, all_preds, average=None, labels=range(num_classes), zero_division=0)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_per_class": f1_per_class,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class
    }

In [ ]:
def compute_confusion_matrix(model, data_loader, device="cpu", num_classes=None):
    model.to(device)
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    if num_classes is None:
        num_classes = max(max(all_labels), max(all_preds)) + 1

    cm = confusion_matrix(all_labels, all_preds)
    return cm

## **4. Fonctions de training et validation**

In [ ]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    all_metrics = compute_all_metrics(all_labels, all_preds, NUM_CLASSES)
    ModelSaver.save_epoch(epoch, all_metrics, total_loss / len(train_loader), mode="train")

    f1_macro = all_metrics["f1_macro"]
    f1_per_class = all_metrics["f1_per_class"]

    # 🔹 Affichage
    print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [ ]:
import torch

def validate(epoch):
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    all_metrics = compute_all_metrics(all_labels, all_preds, NUM_CLASSES)
    ModelSaver.save_epoch(epoch, all_metrics, total_loss / len(val_loader), mode="val")

    f1_macro = all_metrics["f1_macro"]
    f1_per_class = all_metrics["f1_per_class"]

    return (total_loss / len(val_loader), f1_macro, f1_per_class, np.array(all_preds), np.array(all_labels))

## **5. Entrainement**

**Entrainement**

In [ ]:
import csv
import pandas as pd
from sklearn.metrics import confusion_matrix

best_f1 = 0.0

csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro',
              'val_loss', 'val_f1_macro']


loss_train, loss_val = [], []

for epoch in range(NUM_EPOCHS):

    # ===== TRAIN =====
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch(epoch)
    loss_train.append(train_loss)

    if scheduler:
        scheduler.step()

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")

    # ===== VALIDATION =====
    val_loss, val_f1_macro, val_f1_per_class, val_preds, val_labels = validate(epoch)
    loss_val.append(val_loss)
    
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val F1 Macro: {val_f1_macro:.4f}")

    if unfreeze:
        unfreeze_progressive(model, epoch)



    # ===== BEST MODEL =====
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro

        ModelSaver.save_model(model, name=f"best_model_epoch_{epoch}.pth")

        cm = compute_confusion_matrix(model, val_loader, device=DEVICE, num_classes=NUM_CLASSES)
        ModelSaver.save_confusion_matrix(cm, name=f"confusion_matrix_epoch_{epoch}.csv")        

        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")


       

100%|██████████| 201/201 [00:48<00:00,  4.18it/s]


Train F1 macro: 0.1159

Epoch 1/25
Train Loss: 2.2086 | Train Acc: 0.4089
Train F1 Macro: 0.1159
Val Loss: 1.2817
Val F1 Macro: 0.2069
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.2069


100%|██████████| 201/201 [00:44<00:00,  4.52it/s]


Train F1 macro: 0.2635

Epoch 2/25
Train Loss: 0.9849 | Train Acc: 0.7243
Train F1 Macro: 0.2635
Val Loss: 0.9451
Val F1 Macro: 0.2878
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.2878


100%|██████████| 201/201 [00:44<00:00,  4.52it/s]


...

Train F1 macro: 0.9928

Epoch 21/25
Train Loss: 0.0261 | Train Acc: 0.9925
Train F1 Macro: 0.9928
Val Loss: 1.1040
Val F1 Macro: 0.4697


100%|██████████| 201/201 [00:43<00:00,  4.57it/s]


Train F1 macro: 0.9963

Epoch 22/25
Train Loss: 0.0247 | Train Acc: 0.9931
Train F1 Macro: 0.9963
Val Loss: 1.1494
Val F1 Macro: 0.5296


100%|██████████| 201/201 [00:44<00:00,  4.52it/s]


Train F1 macro: 0.9967

Epoch 23/25
Train Loss: 0.0163 | Train Acc: 0.9961
Train F1 Macro: 0.9967


In [ ]:
plt.plot(loss_train, 'r' , marker='o', linestyle='-', label = 'loss_train')
plt.plot(loss_val, 'b' , marker='o', linestyle='-', label = 'loss_test')
plt.xlabel('Epoch')
plt.ylabel('Loss fonction')
plt.grid()
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


 
# 1. Tes données (extraites de ton image)
df = pd.read_csv('../models/training_logefficientnetb0_audrey_better_loader_25epoch.csv')
print(df.head())
epochs = df['epoch']
train_loss = df['train_loss']
val_loss = df['val_loss']
train_f1 = df['train_f1_macro']
val_f1 = df['val_f1_macro']


 
# 2. Création de la figure avec deux sous-graphiques (côte à côte)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))


 
# --- Graphique 1 : La Loss ---
ax1.plot(epochs, train_loss, label='Train Loss', marker='o', linestyle='--')
ax1.plot(epochs, val_loss, label='Val Loss', marker='o')
ax1.set_title('Évolution de la Perte (Loss)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_xticks(epochs) # Pour n'afficher que des entiers sur l'axe X
ax1.legend()
ax1.grid(True, alpha=0.3)


 
# --- Graphique 2 : Le F1-Score ---
ax2.plot(epochs, train_f1, label='Train F1 Macro', marker='s', color='green', linestyle='--')
ax2.plot(epochs, val_f1, label='Val F1 Macro', marker='s', color='orange')
ax2.set_title('Évolution du F1-Score Macro')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.set_xticks(epochs)
ax2.legend()
ax2.grid(True, alpha=0.3)


 
plt.tight_layout()
plt.show()